> **Quick start — click *Run All* — no setup needed.**
> Cached PNG figures are displayed by default (`RERUN = False`).
> Set `RERUN = True` and re-run to regenerate figures from the source scripts.


In [ ]:
import subprocess, sys, pathlib, importlib
from IPython.display import Image, display

# ── locate repository root (search upward for lunar/__init__.py) ──────────
_here = pathlib.Path.cwd()
REPO = None
for _p in [_here, *_here.parents]:
    if (_p / "lunar" / "__init__.py").exists():
        REPO = _p
        break
if REPO is None:
    raise RuntimeError("Cannot find REPO root — run from inside Lunar-V2/")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

FIGS    = REPO / "output" / "figures"
SCRIPTS = REPO / "scripts" / "phase2"


def show_figure(name: str, caption: str = "") -> None:
    """Display a cached PNG from output/figures/."""
    path = FIGS / name
    if not path.exists():
        print(f"[warn] figure not found: {path}")
        return
    display(Image(str(path), width=820))
    if caption:
        from IPython.display import Markdown
        display(Markdown(f"*{caption}*"))


def run_script(rel: str) -> None:
    """Run a Phase-2 script via subprocess, streaming stdout."""
    script = SCRIPTS / rel
    print(f"▶  running {script.relative_to(REPO)} …")
    result = subprocess.run(
        [sys.executable, str(script)],
        cwd=str(REPO),
        capture_output=False,
    )
    if result.returncode != 0:
        print(f"[error] script exited with code {result.returncode}")
    else:
        print("[done]")

print(f"REPO  = {REPO}")
print(f"FIGS  = {FIGS}  (exists: {FIGS.exists()})")
print(f"SCRIPTS = {SCRIPTS}  (exists: {SCRIPTS.exists()})")


# Phase 2 (Global) — K(T,ρ) model + flat-terrain validation

Replicates the upstream `Global/` MATLAB results from Martinez & Siegler (2021).
Covers the conductivity–temperature overlay (Fig 1) and the 45 °N highlands
diurnal validation against Diviner T7 (Fig 2).


In [ ]:
RERUN = False  # Fig 2 takes ~3 min (80-lunation spin-up)


## §1 — Figure 1 · K(T,ρ) overlay

Plots the Hayne (2017) χ–T³ conductivity model against the Martinez & Siegler
(2021) polynomial at **six bulk densities** (ρ = 1100 – 2100 kg m⁻³).

Key details:
- Hayne B₁ = 2.0022 × 10⁻¹³ (patched from erroneous prior value).
- Low-T divergence visible below ~100 K — Hayne's radiative term collapses
  while M&S retains a shallow positive slope.
- Mirrors the `Global/StandardModel` vs `Global/UpdatedModel` comparison in
  the upstream MATLAB workspace.


In [ ]:
if RERUN:
    run_script("global/fig1_K_vs_T.py")

show_figure(
    "phase2_fig1_K_vs_T_multi_rho.png",
    "Fig 1 — K(T) at six bulk densities (Hayne 2017 vs Martinez & Siegler 2021)",
)


## §2 — Figure 2 · 45 °N highlands diurnal vs Diviner T7

Flat-terrain 1-D heat-flow simulation at 45 °N, one full lunation
(~708.7 h).

Key modelling choices:
- **Albedo** — Vasavada (2012) angle-dependent formula:
  A(i) = A₀ + 0.06·i³ + 0.25·i⁸ (incidence angle i in units of π/2),
  baked into the insolation array before calling the solver.
- **Density profile** — Hayne (2017) ρ(z) two-layer exponential.
- **Spin-up** — `spinup_depth_m = 0.10 m` (top-only convergence criterion).

RMSE vs Diviner T7 (40–50 °N band):

| Model | Full diurnal RMSE | Night-only RMSE |
|-------|:-----------------:|:---------------:|
| Hayne 2017 | 9.2 K | 4.9 K |
| Martinez & Siegler 2021 | 9.6 K | 6.2 K |


In [ ]:
if RERUN:
    print("~3 min — 80-lunation spin-up…")
    run_script("global/fig2_diurnal_45N.py")

show_figure(
    "phase2_fig2_diurnal_45N.png",
    "Fig 2 — 45 °N diurnal cycle, Hayne vs M&S, Diviner T7 overlay",
)
